## Cell 1 — Imports

In [45]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import (
    GINEConv, GATv2Conv, TransformerConv, GCNConv,
    global_mean_pool, global_max_pool, GraphNorm, GlobalAttention)
from torch.optim import AdamW
from torch.optim.lr_scheduler import (
    CosineAnnealingWarmRestarts, LinearLR, SequentialLR)
from torch.utils.data import WeightedRandomSampler

import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, matthews_corrcoef, recall_score,
    confusion_matrix, balanced_accuracy_score, f1_score, precision_score,
    r2_score, mean_squared_error, mean_absolute_error,
    roc_curve, precision_recall_curve, average_precision_score)
from sklearn.model_selection import KFold, train_test_split
from sklearn.manifold import TSNE
from scipy import stats
from rdkit import RDLogger, Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
import warnings, copy, os, pickle

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')
print("✅ Imports OK")


✅ Imports OK


## Cell 2 — Config & Directories
> **Set `CKPT_DIR_OVERRIDE` to your checkpoint folder path.**

In [46]:
import torch, numpy as np, os, matplotlib.pyplot as plt, seaborn as sns

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── SET THIS to your checkpoint folder ────────────────────────────────────────
CKPT_DIR_OVERRIDE = r"D:\Research Papers\Actual Biotechnology\checkpoints_v12"
# ─────────────────────────────────────────────────────────────────────────────

def find_ckpt_dir():
    if CKPT_DIR_OVERRIDE and os.path.isdir(CKPT_DIR_OVERRIDE):
        n = len([f for f in os.listdir(CKPT_DIR_OVERRIDE) if f.endswith(".pkl")])
        print(f"  ✅ Checkpoint dir: {CKPT_DIR_OVERRIDE}  ({n} pkl files)")
        return CKPT_DIR_OVERRIDE
    candidates = [
        r"D:\Research Papers\Actual Biotechnology\checkpoints_v12",
        r"D:\Research Papers\Actual Biotechnology\checkpoints_v13",
        r"D:\checkpoints_v12", r"D:\checkpoints_v13",
        "./checkpoints_v12", "./checkpoints_v13",
        "/content/checkpoints_v12", "/content/checkpoints_v13",
        "/kaggle/working/checkpoints_v12",
    ]
    for c in candidates:
        if os.path.isdir(c) and any(f.endswith(".pkl") for f in os.listdir(c)):
            n = len([f for f in os.listdir(c) if f.endswith(".pkl")])
            print(f"  ✅ Found: {c}  ({n} pkl files)")
            return c
    base = "."
    d = os.path.join(base, "checkpoints_v14")
    os.makedirs(d, exist_ok=True)
    print(f"  ⚠️  No checkpoints found — new ones will save to: {d}")
    return d

CKPT_DIR = find_ckpt_dir()

# ── Figure & data paths ───────────────────────────────────────────────────────
FIG_DIR  = os.path.join(r"D:\Research Papers\Actual Biotechnology", "figures_v14")
os.makedirs(FIG_DIR, exist_ok=True)
print(f"  Figures → {FIG_DIR}")

DATA_PATH_DEFAULT = r"D:\Research Papers\Actual Biotechnology\WholedatasetGAN.csv"
EXT_PATH_DEFAULT  = r"D:\Research Papers\Actual Biotechnology\FDAdrugsGNN.csv"

# ── Palettes ──────────────────────────────────────────────────────────────────
MODEL_NAMES     = ["GCN", "GAT", "GT", "MSMP"]
MODEL_COLORS    = ["#E74C3C", "#3498DB", "#2ECC71", "#9B59B6"]
COND_COLORS     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
COND_LABELS     = ["Random\nNoAug", "Random\nAug", "Scaffold\nNoAug", "Scaffold\nAug"]
COND_NAMES_LONG = ["Random | No Aug", "Random | Aug", "Scaffold | No Aug", "Scaffold | Aug"]
BRIGHT_SEEDS    = ["#FF4757", "#2ED573", "#1E90FF", "#FFA502", "#FF6B81"]

sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams.update({"figure.dpi": 200, "savefig.dpi": 200, "font.family": "DejaVu Sans"})

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  [saved] {path}")

print("✅ Config ready")


Device: cuda
  ✅ Checkpoint dir: D:\Research Papers\Actual Biotechnology\checkpoints_v12  (16 pkl files)
  Figures → D:\Research Papers\Actual Biotechnology\figures_v14
✅ Config ready


## Cell 3 — Molecular Feature Functions

In [47]:
def enhanced_atom_features(atom):
    atom_types = ['C','N','O','S','F','Cl','Br','I','P','B']
    f = [1. if atom.GetSymbol()==t else 0. for t in atom_types]
    f += [atom.GetAtomicNum()/100., atom.GetDegree()/8.,
          atom.GetFormalCharge()/5., atom.GetTotalNumHs()/8.,
          atom.GetTotalValence()/8., float(atom.GetIsAromatic()),
          float(atom.IsInRing()),
          float(atom.GetChiralTag()!=Chem.rdchem.ChiralType.CHI_UNSPECIFIED)]
    hyb = atom.GetHybridization()
    for ht in [Chem.rdchem.HybridizationType.SP,
               Chem.rdchem.HybridizationType.SP2,
               Chem.rdchem.HybridizationType.SP3]:
        f.append(1. if hyb==ht else 0.)
    return f  # 21-dim

def enhanced_bond_features(bond):
    bt = bond.GetBondType()
    return [float(bt==Chem.rdchem.BondType.SINGLE),
            float(bt==Chem.rdchem.BondType.DOUBLE),
            float(bt==Chem.rdchem.BondType.TRIPLE),
            float(bt==Chem.rdchem.BondType.AROMATIC),
            float(bond.GetIsConjugated()),
            float(bond.IsInRing()),
            float(bond.GetStereo()!=Chem.rdchem.BondStereo.STEREONONE)]  # 7-dim

def smiles_to_graph(smiles, label, pic50):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x  = [enhanced_atom_features(a) for a in mol.GetAtoms()]
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = enhanced_bond_features(bond)
        ei += [[i,j],[j,i]]; ea += [bf, bf]
    if not ei:
        ei = [[0,0],[0,0]]; ea = [[0.]*7, [0.]*7]
    try:
        desc = [
            Descriptors.MolWt(mol)/1000.,      Descriptors.MolLogP(mol)/10.,
            Descriptors.TPSA(mol)/200.,         Descriptors.NumRotatableBonds(mol)/20.,
            QED.qed(mol),                       Descriptors.NumHDonors(mol)/10.,
            Descriptors.NumHAcceptors(mol)/10., float(rdMolDescriptors.CalcNumAromaticRings(mol))/5.,
            Descriptors.FractionCSP3(mol),      float(mol.GetNumHeavyAtoms())/50.,
            float(rdMolDescriptors.CalcNumRings(mol))/10.,
            min(Descriptors.BertzCT(mol)/1000., 3.),
            float(rdMolDescriptors.CalcNumHeteroatoms(mol))/20.]
    except:
        desc = [0.]*13
    return Data(
        x          = torch.tensor(x,  dtype=torch.float),
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous(),
        edge_attr  = torch.tensor(ea, dtype=torch.float),
        desc       = torch.tensor(desc, dtype=torch.float).view(1,-1),
        y_cls      = torch.tensor([label], dtype=torch.float),
        y_reg      = torch.tensor([pic50], dtype=torch.float),
        smiles     = smiles)
print("✅ Feature functions defined  (node_dim=21, edge_dim=7, desc_dim=13)")


✅ Feature functions defined  (node_dim=21, edge_dim=7, desc_dim=13)


## Cell 4 — Load Dataset

In [48]:
DATA_PATHS = [
    DATA_PATH_DEFAULT,
    r"D:\Biotechnology Research\WholedatasetGAN.csv",
    "./WholedatasetGAN.csv",
    "/content/WholedatasetGAN.csv",
    "/kaggle/input/anti-inflammatory/WholedatasetGAN.csv"]
DATA_PATH = next((p for p in DATA_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("WholedatasetGAN.csv not found — update DATA_PATH_DEFAULT in Cell 2")
print(f"Dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["active"] = (df["pIC50"] >= 6).astype(int)
df = df.dropna(subset=["pIC50"]).drop_duplicates(subset="SMILES").reset_index(drop=True)

graphs, valid_smiles = [], []
for _, row in df.iterrows():
    g = smiles_to_graph(row.SMILES, row.active, row.pIC50)
    if g is not None:
        graphs.append(g); valid_smiles.append(row.SMILES)

df_filtered = df[df["SMILES"].isin(valid_smiles)].reset_index(drop=True)
y_all = np.array([g.y_reg.item() for g in graphs])
y_reg_mean = float(y_all.mean())
y_reg_std  = float(y_all.std() + 1e-8)
for g in graphs:
    g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std

NODE_DIM = graphs[0].x.shape[1]
DESC_DIM = graphs[0].desc.shape[1]
cc = df_filtered["active"].value_counts()
n_pos = cc.get(1, 1); n_neg = cc.get(0, 1)
pos_weight_val = len(df_filtered) / (2. * n_pos)
neg_weight_val = len(df_filtered) / (2. * n_neg)

print(f"Graphs : {len(graphs)}   node_dim={NODE_DIM}   desc_dim={DESC_DIM}")
print(f"Classes: {cc.to_dict()}")
print(f"pIC50  : mean={y_reg_mean:.3f}   std={y_reg_std:.3f}")


Dataset: D:\Biotechnology Research\WholedatasetGAN.csv
Graphs : 4300   node_dim=21   desc_dim=13
Classes: {0: 2150, 1: 2150}
pIC50  : mean=6.408   std=1.208


## Cell 5 — Chemical Space Analysis (Fig 01)

In [49]:
mols_cs = [Chem.MolFromSmiles(s) for s in df_filtered["SMILES"] if Chem.MolFromSmiles(s)]
fps_cs  = [AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) for m in mols_cs[:1000]]
sims_cs = [DataStructs.TanimotoSimilarity(fps_cs[i], fps_cs[j])
           for i in range(len(fps_cs)) for j in range(i+1, len(fps_cs))]
mean_sim = float(np.mean(sims_cs))
print(f"Tanimoto  mean={mean_sim:.4f}   std={np.std(sims_cs):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sims_cs, bins=60, color="#4C72B0", edgecolor="white", alpha=0.87)
axes[0].axvline(mean_sim, color="red", linestyle="--", lw=1.8, label=f"Mean={mean_sim:.3f}")
axes[0].set_xlabel("Tanimoto Similarity"); axes[0].set_ylabel("Frequency")
axes[0].set_title("Pairwise Tanimoto Distribution"); axes[0].legend()

axes[1].hist(df_filtered["pIC50"], bins=50, color="#55A868", edgecolor="white", alpha=0.87)
axes[1].axvline(6., color="red", linestyle="--", lw=1.8, label="Threshold=6.0")
axes[1].set_xlabel("pIC50"); axes[1].set_ylabel("Count")
axes[1].set_title("pIC50 Distribution"); axes[1].legend()

plt.suptitle("Chemical Space Analysis", fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout(); savefig("01_chemical_space.png")


Tanimoto  mean=0.1170   std=0.0619
  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\01_chemical_space.png


## Cell 6 — Augmentation, TTA, Utilities

In [50]:
N_AUG = 3
N_TTA = 3

# ── Graph-level augmentation ──────────────────────────────────────────────────
def edge_dropout(data, p=0.15):
    data = data.clone()
    n = data.edge_index.size(1) // 2
    if n == 0: return data
    mask = (torch.rand(n, device=data.x.device) > p).repeat_interleave(2)
    data.edge_index = data.edge_index[:, mask]
    data.edge_attr  = data.edge_attr[mask]
    return data

def node_dropout(data, p=0.1):
    data = data.clone()
    data.x = data.x * (torch.rand(data.x.size(0), 1, device=data.x.device) > p)
    return data

def smiles_augmentation(smiles, label, pic50, n=N_AUG):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return []
    out = []
    for _ in range(n):
        try:
            g = smiles_to_graph(
                Chem.MolToSmiles(mol, doRandom=True, canonical=False), label, pic50)
            if g:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                out.append(g)
        except: pass
    return out

print(f"Pre-computing SMILES augmentation cache (n_aug={N_AUG})…")
aug_cache = {
    i: smiles_augmentation(
        df_filtered.iloc[i]["SMILES"],
        df_filtered.iloc[i]["active"],
        df_filtered.iloc[i]["pIC50"])
    for i in range(len(graphs))}
print(f"Cache ready: {sum(len(v) for v in aug_cache.values())} augmented graphs")

# ── Test-time augmentation ────────────────────────────────────────────────────
def tta_predict(model, smiles_list, labels_cls, labels_reg,
                device, n_tta=N_TTA, batch_size=192):
    model.eval()
    all_p = [[] for _ in smiles_list]
    all_r = [[] for _ in smiles_list]
    for t in range(n_tta + 1):
        tg, ti = [], []
        for i, (sm, lc, lr) in enumerate(zip(smiles_list, labels_cls, labels_reg)):
            mol = Chem.MolFromSmiles(sm)
            if t == 0:
                g = smiles_to_graph(sm, int(lc), float(lr))
            elif mol:
                g = smiles_to_graph(
                    Chem.MolToSmiles(mol, doRandom=True, canonical=False), int(lc), float(lr))
            else: continue
            if g:
                g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
                tg.append(g); ti.append(i)
        ptr = 0
        with torch.no_grad():
            for batch in DataLoader(tg, batch_size, shuffle=False, num_workers=0):
                batch = batch.to(device)
                co, ro = model(batch)
                ps = torch.sigmoid(co).cpu().numpy()
                rs = ro.cpu().numpy()
                for k in range(len(ps)):
                    all_p[ti[ptr+k]].append(float(ps[k]))
                    all_r[ti[ptr+k]].append(float(rs[k]))
                ptr += len(ps)
    return (np.array([np.mean(p) if p else .5 for p in all_p]),
            np.array([np.mean(r) if r else .0 for r in all_r]))

# ── Weighted sampler ──────────────────────────────────────────────────────────
def make_weighted_sampler(graph_list):
    w = torch.tensor(
        [pos_weight_val if int(g.y_cls.item())==1 else neg_weight_val
         for g in graph_list], dtype=torch.float)
    return WeightedRandomSampler(w, len(w), replacement=True)

# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=.6, gamma=2., ls=.03):
        super().__init__()
        self.a = alpha; self.g = gamma; self.ls = ls
    def forward(self, logits, targets):
        if self.ls > 0:
            targets = targets * (1 - self.ls) + .5 * self.ls
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        return (self.a * (1 - torch.exp(-bce)) ** self.g * bce).mean()

print("✅ Augmentation, TTA, FocalLoss, sampler ready")


Pre-computing SMILES augmentation cache (n_aug=3)…
Cache ready: 12900 augmented graphs
✅ Augmentation, TTA, FocalLoss, sampler ready


## Cell 7 — MSMP Model

In [51]:
class HighPerfMSMP(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=192, heads=4,
                 dropout=.21, num_layers=4, **kw):
        super().__init__()
        self.node_embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.edge_proj_gine = nn.Linear(7, hidden)
        self.edge_proj_attn = nn.Linear(7, 32)
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        self.layers = nn.ModuleList()
        self.residual_scales = nn.ParameterList()
        self.num_layers = num_layers
        for i in range(num_layers):
            dp = dropout * (1 + .1*i)
            self.layers.append(nn.ModuleDict({
                "gine":  GINEConv(nn.Sequential(
                    nn.Linear(hidden, hidden*2), nn.GELU(),
                    nn.Dropout(dp), nn.Linear(hidden*2, hidden))),
                "gat":   GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32),
                "trans": TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32),
                "norm":  GraphNorm(hidden)}))
            self.residual_scales.append(nn.Parameter(torch.tensor(.5)))
        self.layer_pool_weights = nn.Parameter(torch.ones(num_layers) / num_layers)
        self.attn_pool = GlobalAttention(nn.Sequential(
            nn.Linear(hidden, hidden//2), nn.ReLU(), nn.Linear(hidden//2, 1)))
        pool_dim = hidden*4 + desc_dim
        self.pool_norm = nn.LayerNorm(pool_dim)
        self.fusion = nn.Sequential(
            nn.Linear(pool_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden//2), nn.GELU(),
            nn.Dropout(dropout*.5), nn.Linear(hidden//2, 128))
        self.cls_head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(.3), nn.Linear(64, 1))
        self.reg_head = nn.Sequential(
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(.3), nn.Linear(64, 1))

    def forward(self, data, return_embedding=False):
        x  = self.node_embed(data.x)
        ei = data.edge_index; ea = data.edge_attr; bat = data.batch
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        eg = self.edge_proj_gine(ea); ea2 = self.edge_proj_attn(ea); lp = []
        for i, layer in enumerate(self.layers):
            h = (layer["gine"](x, ei, eg) +
                 layer["gat"](x, ei, ea2) +
                 layer["trans"](x, ei, ea2))
            h = layer["norm"](h, bat)
            x = F.gelu(x + torch.sigmoid(self.residual_scales[i]) * h)
            lp.append(global_mean_pool(h, bat))
        lw = F.softmax(self.layer_pool_weights, 0)
        g = torch.cat([
            global_mean_pool(x, bat), global_max_pool(x, bat),
            self.attn_pool(x, bat),
            (torch.stack(lp, 1) * lw.view(1,-1,1)).sum(1), desc], -1)
        f = self.fusion(self.pool_norm(g))
        if return_embedding: return f
        return self.cls_head(f).squeeze(-1), self.reg_head(f).squeeze(-1)
print("✅ MSMP defined")


✅ MSMP defined


## Cell 8 — GCN Model

In [52]:
class SimpleGCN(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=128,
                 dropout=.2, num_layers=4, **kw):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.convs = nn.ModuleList([GCNConv(hidden, hidden) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        p = hidden*2 + desc_dim
        self.cls_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))

    def forward(self, data, return_embedding=False):
        x = self.embed(data.x)
        for conv, norm in zip(self.convs, self.norms):
            x = x + norm(F.gelu(conv(x, data.edge_index)))
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        g = torch.cat([global_mean_pool(x, data.batch),
                       global_max_pool(x, data.batch), desc], -1)
        if return_embedding: return g
        return self.cls_head(g).squeeze(-1), self.reg_head(g).squeeze(-1)
print("✅ GCN defined")


✅ GCN defined


## Cell 9 — GAT Model

In [53]:
class SimpleGAT(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=128,
                 heads=4, dropout=.2, num_layers=4, **kw):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.eproj = nn.Linear(7, 32)
        self.convs = nn.ModuleList([
            GATv2Conv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
            for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        p = hidden*2 + desc_dim
        self.cls_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))

    def forward(self, data, return_embedding=False):
        x  = self.embed(data.x)
        ea = self.eproj(data.edge_attr)
        for conv, norm in zip(self.convs, self.norms):
            x = x + norm(F.gelu(conv(x, data.edge_index, ea)))
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        g = torch.cat([global_mean_pool(x, data.batch),
                       global_max_pool(x, data.batch), desc], -1)
        if return_embedding: return g
        return self.cls_head(g).squeeze(-1), self.reg_head(g).squeeze(-1)
print("✅ GAT defined")


✅ GAT defined


## Cell 10 — Graph Transformer Model

In [54]:
class SimpleGT(nn.Module):
    def __init__(self, node_dim, desc_dim=13, hidden=128,
                 heads=4, dropout=.2, num_layers=4, **kw):
        super().__init__()
        self.embed = nn.Sequential(
            nn.Linear(node_dim, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(dropout))
        self.eproj = nn.Linear(7, 32)
        self.convs = nn.ModuleList([
            TransformerConv(hidden, hidden, heads=heads, concat=False, edge_dim=32)
            for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(num_layers)])
        self.desc_bn = nn.BatchNorm1d(desc_dim)
        p = hidden*2 + desc_dim
        self.cls_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))
        self.reg_head = nn.Sequential(
            nn.Linear(p,128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128,64), nn.GELU(), nn.Dropout(dropout*.5), nn.Linear(64,1))

    def forward(self, data, return_embedding=False):
        x  = self.embed(data.x)
        ea = self.eproj(data.edge_attr)
        for conv, norm in zip(self.convs, self.norms):
            x = x + norm(F.gelu(conv(x, data.edge_index, ea)))
        desc = data.desc
        if desc.size(0) > 1: desc = self.desc_bn(desc)
        g = torch.cat([global_mean_pool(x, data.batch),
                       global_max_pool(x, data.batch), desc], -1)
        if return_embedding: return g
        return self.cls_head(g).squeeze(-1), self.reg_head(g).squeeze(-1)
print("✅ Graph Transformer defined")


✅ Graph Transformer defined


## Cell 11 — Training Infrastructure

In [55]:
def build_scheduler(opt, lr, warmup=5):
    return SequentialLR(opt, schedulers=[
        LinearLR(opt, start_factor=.1, end_factor=1., total_iters=warmup),
        CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2)],
        milestones=[warmup])

def train_epoch(model, loader, opt, sched, hp, device,
                epoch, verbose=True, use_aug=True):
    model.train(); total = 0.
    cls_fn = FocalLoss(alpha=hp["focal_alpha"], gamma=hp["focal_gamma"],
                       ls=hp.get("label_smoothing", .03))
    for batch in loader:
        batch = batch.to(device)
        if use_aug:
            if torch.rand(1) > .3:
                batch = edge_dropout(batch, hp["edge_dropout"])
            batch = node_dropout(batch, hp["node_dropout"])
        opt.zero_grad()
        co, ro = model(batch)
        loss = (cls_fn(co, batch.y_cls.squeeze()) +
                hp["reg_weight"] * F.huber_loss(
                    ro, batch.y_reg_norm.squeeze(), delta=1.))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        opt.step(); total += loss.item()
    sched.step()
    if verbose and (epoch % 5 == 0 or epoch == 0):
        print(f"    ep{epoch:3d}  loss={total/len(loader):.4f}"
              f"  lr={opt.param_groups[0]['lr']:.2e}")
    return total / len(loader)

def get_splits(df, graphs, split_type, n_splits=5, seed=42):
    if split_type == "random":
        return [train_test_split(
            range(len(graphs)), train_size=.8,
            random_state=seed+i, stratify=df["active"].values)
            for i in range(n_splits)]
    df_s = df.copy()
    df_s["scaffold"] = df_s["SMILES"].apply(
        lambda s: MurckoScaffold.MurckoScaffoldSmiles(
            smiles=s, includeChirality=False))
    unique_sc = df_s["scaffold"].unique()
    splits = []
    for tr, te in KFold(n_splits=n_splits, shuffle=True,
                        random_state=seed).split(unique_sc):
        tr_set = set(unique_sc[tr]); te_set = set(unique_sc[te])
        splits.append((
            df_s.index[df_s["scaffold"].isin(tr_set)].tolist(),
            df_s.index[df_s["scaffold"].isin(te_set)].tolist()))
    return splits

def fold_similarity(tr_sm, te_sm):
    def fps(sms):
        return [AllChem.GetMorganFingerprintAsBitVect(
                    Chem.MolFromSmiles(s), 2, 2048)
                for s in sms if Chem.MolFromSmiles(s)]
    tf, tf2 = fps(tr_sm), fps(te_sm)
    if not tf or not tf2: return 0., 0.
    mx = [max(DataStructs.TanimotoSimilarity(t, r) for r in tf) for t in tf2]
    return float(np.mean(mx)), float(np.std(mx))

print("✅ Training infrastructure ready")


✅ Training infrastructure ready


## Cell 12 — Hyperparameters

In [56]:
# MSMP — Optuna-tuned (v8)
best_hparams = {
    "lr":               0.0007175353182914354,
    "hidden":           192,
    "heads":            4,
    "dropout":          0.20789169446550804,
    "weight_decay":     2.7145406595506218e-05,
    "edge_dropout":     0.1826073098110208,
    "node_dropout":     0.05256789924774226,
    "focal_alpha":      0.4528973861125163,
    "focal_gamma":      2.5959215680388827,
    "reg_weight":       0.2345157697916489,
    "label_smoothing":  0.012352647462804418}

# GCN / GAT / GT — shared baseline hparams
base_hparams = {
    "lr":               0.001,
    "hidden":           128,
    "heads":            4,
    "dropout":          0.20,
    "weight_decay":     1e-4,
    "edge_dropout":     0.15,
    "node_dropout":     0.05,
    "focal_alpha":      0.50,
    "focal_gamma":      2.0,
    "reg_weight":       0.25,
    "label_smoothing":  0.02}

MODEL_CFG = {
    "GCN":  (SimpleGCN,    base_hparams),
    "GAT":  (SimpleGAT,    base_hparams),
    "GT":   (SimpleGT,     base_hparams),
    "MSMP": (HighPerfMSMP, best_hparams)}

print("Model configs:")
for k, (cls, hp) in MODEL_CFG.items():
    print(f"  {k:<5}  hidden={hp['hidden']}  lr={hp['lr']:.4f}"
          f"  dropout={hp['dropout']:.2f}")


Model configs:
  GCN    hidden=128  lr=0.0010  dropout=0.20
  GAT    hidden=128  lr=0.0010  dropout=0.20
  GT     hidden=128  lr=0.0010  dropout=0.20
  MSMP   hidden=192  lr=0.0007  dropout=0.21


## Cell 13 — CV Runner & Checkpoint System

In [57]:
EPOCHS  = 60
PATIENCE = 10

def run_cv(split_type, use_aug, model_class, hparams, label,
           n_folds=5, n_ensemble=None):
    if n_ensemble is None:
        n_ensemble = 3 if use_aug else 2
    print(f"\n{'='*76}")
    print(f"  [{label}]  split={split_type.upper()}  aug={'YES' if use_aug else 'NO'}"
          f"  model={model_class.__name__}  ens={n_ensemble}")
    print(f"{'='*76}")

    splits = get_splits(df_filtered, graphs, split_type, n_folds)
    all_cls, all_reg, all_sim, all_pl, all_hist = [], [], [], [], []

    for fold, (tr_idx, te_idx) in enumerate(splits):
        print(f"\n  --- Fold {fold+1}/{n_folds} ---")
        tr_g = [graphs[i] for i in tr_idx]
        if use_aug:
            for i in tr_idx: tr_g.extend(aug_cache[i])
        te_g = [graphs[i] for i in te_idx]

        sm, ss = fold_similarity(
            [df_filtered["SMILES"].iloc[i] for i in tr_idx],
            [df_filtered["SMILES"].iloc[i] for i in te_idx])
        all_sim.append((sm, ss))
        print(f"  Sim={sm:.4f}±{ss:.4f}  Train={len(tr_g)}  Test={len(te_g)}")

        tr_ldr = DataLoader(tr_g, 192, sampler=make_weighted_sampler(tr_g),
                            num_workers=0, pin_memory=True)
        te_ldr = DataLoader(te_g, 192, shuffle=False,
                            num_workers=0, pin_memory=True)

        ens_p, ens_r, fold_hist = [], [], []

        for seed in [42, 123, 777, 2024, 31415][:n_ensemble]:
            torch.manual_seed(seed)
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

            mkw = {"hidden":  hparams["hidden"],
                   "heads":   hparams.get("heads", 4),
                   "dropout": hparams["dropout"]}
            model = model_class(NODE_DIM, DESC_DIM, **mkw).to(device)
            opt   = AdamW(model.parameters(), lr=hparams["lr"],
                          weight_decay=hparams["weight_decay"])
            sched = build_scheduler(opt, hparams["lr"])
            best_acc, best_state, best_thr, wait = 0., None, .5, 0
            hist = []

            for epoch in range(EPOCHS):
                tl = train_epoch(model, tr_ldr, opt, sched, hparams,
                                 device, epoch, verbose=(fold==0), use_aug=use_aug)
                model.eval(); rp, rl = [], []
                with torch.no_grad():
                    for b in te_ldr:
                        b = b.to(device); co, _ = model(b)
                        rp.extend(torch.sigmoid(co).cpu().numpy())
                        rl.extend(b.y_cls.cpu().numpy())
                rp = np.array(rp); rl = np.array(rl)
                th = np.linspace(.1, .9, 161)
                ac = [accuracy_score(rl, (rp>=t).astype(int)) for t in th]
                va = float(max(ac)); vt = float(th[np.argmax(ac)])
                try:   vauc = float(roc_auc_score(rl, rp))
                except: vauc = .5
                hist.append((epoch, tl, va))

                if va > best_acc:
                    best_acc  = va
                    best_state = copy.deepcopy(model.state_dict())
                    best_thr  = vt; wait = 0
                    print(f"    seed={seed} ep{epoch:3d}"
                          f"  ACC={va:.4f}  AUC={vauc:.4f}  thr={vt:.3f}")
                else:
                    wait += 1
                    if wait >= PATIENCE:
                        print(f"    seed={seed} early-stop ep{epoch}"
                              f"  best={best_acc:.4f}"); break

            fold_hist.append((seed, hist))
            if best_state: model.load_state_dict(best_state)

            te_sm = [df_filtered["SMILES"].iloc[i] for i in te_idx]
            if use_aug:
                ap, ar = tta_predict(
                    model, te_sm,
                    [graphs[i].y_cls.item() for i in te_idx],
                    [graphs[i].y_reg.item() for i in te_idx], device)
            else:
                model.eval(); fp, fr = [], []
                with torch.no_grad():
                    for b in DataLoader(te_g, 192, shuffle=False, num_workers=0):
                        b = b.to(device); co, ro = model(b)
                        fp.extend(torch.sigmoid(co).cpu().numpy())
                        fr.extend(ro.cpu().numpy())
                ap, ar = np.array(fp), np.array(fr)

            ens_p.append(ap); ens_r.append(ar)
        all_hist.append(fold_hist)

        # ── Fold-level metrics ────────────────────────────────────────────────
        avg_p  = np.mean(ens_p, 0)
        avg_r  = np.mean(ens_r, 0) * y_reg_std + y_reg_mean
        labels = np.array([graphs[i].y_cls.item() for i in te_idx])
        true_r = np.array([graphs[i].y_reg.item() for i in te_idx])

        th2 = np.linspace(.1, .9, 161)
        ac2 = [accuracy_score(labels, (avg_p>=t).astype(int)) for t in th2]
        preds = (avg_p >= th2[np.argmax(ac2)]).astype(int)
        tn, fp_, fn, tp = confusion_matrix(labels, preds).ravel()

        cls_m = {
            "accuracy":          accuracy_score(labels, preds),
            "auc":               roc_auc_score(labels, avg_p),
            "mcc":               matthews_corrcoef(labels, preds),
            "sensitivity":       recall_score(labels, preds),
            "specificity":       tn/(tn+fp_),
            "balanced_accuracy": balanced_accuracy_score(labels, preds),
            "f1":                f1_score(labels, preds),
            "precision":         precision_score(labels, preds)}
        reg_m = {
            "r2":   r2_score(true_r, avg_r),
            "rmse": float(np.sqrt(mean_squared_error(true_r, avg_r))),
            "mae":  float(mean_absolute_error(true_r, avg_r))}

        print(f"\n  Fold {fold+1}: ACC={cls_m['accuracy']:.4f}"
              f"  AUC={cls_m['auc']:.4f}"
              f"  MCC={cls_m['mcc']:.4f}"
              f"  R²={reg_m['r2']:.4f}")
        all_cls.append(cls_m); all_reg.append(reg_m)

        # ── Train eval (for paper tables) — fd[7] stored already denormalized ─
        plain_tr = [graphs[i] for i in tr_idx]
        model.eval(); tp2, tr2 = [], []
        with torch.no_grad():
            for b in DataLoader(plain_tr, 192, shuffle=False, num_workers=0):
                b = b.to(device); co, ro = model(b)
                tp2.extend(torch.sigmoid(co).cpu().numpy())
                tr2.extend(ro.cpu().numpy())
        all_pl.append((
            avg_p, labels, true_r, avg_r,           # test: probs, cls, reg_true, reg_pred
            np.array(tp2),                           # train probs
            np.array([graphs[i].y_cls.item() for i in tr_idx]),  # train cls labels
            np.array([graphs[i].y_reg.item() for i in tr_idx]),  # train reg true
            np.array(tr2) * y_reg_std + y_reg_mean)) # train reg pred (denormalized)

    def ms(rs):
        return {k: {"mean": float(np.mean([r[k] for r in rs])),
                    "std":  float(np.std([r[k]  for r in rs]))}
                for k in rs[0]}

    cs, rs = ms(all_cls), ms(all_reg)
    print(f"\n  SUMMARY [{label}]")
    for m in ["accuracy","auc","mcc","sensitivity","specificity",
              "f1","precision","balanced_accuracy"]:
        print(f"    {m:<22} {cs[m]['mean']:.4f} ± {cs[m]['std']:.4f}")
    for m in ["r2","rmse","mae"]:
        print(f"    {m:<22} {rs[m]['mean']:.4f} ± {rs[m]['std']:.4f}")
    return all_cls, all_reg, cs, rs, all_pl, all_sim, all_hist


def load_or_run(ckpt_name, split_type, use_aug,
                model_class, hparams, label):
    path = os.path.join(CKPT_DIR, f"{ckpt_name}.pkl")
    if os.path.exists(path):
        print(f"\n✅ CHECKPOINT FOUND: {label}")
        with open(path, "rb") as f:
            return pickle.load(f)
    print(f"\n🚀 RUNNING: {label}")
    result = run_cv(split_type, use_aug, model_class, hparams, label)
    with open(path, "wb") as f:
        pickle.dump(result, f)
    print(f"\n💾 SAVED: {path}")
    return result

print("✅ CV runner and checkpoint system ready")


✅ CV runner and checkpoint system ready


## Part 6 — Training Runs
> Each cell is independent. Loads from checkpoint if available — otherwise trains and saves.

### MSMP | Random | No Aug

In [58]:
r_c1_random_noaug = load_or_run(
    "c1_random_noaug", "random", False,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Random | No Aug")
c1_random_noaug_cls,c1_random_noaug_reg,c1_random_noaug_cls_s,c1_random_noaug_reg_s,c1_random_noaug_pl,c1_random_noaug_sim,c1_random_noaug_hist = r_c1_random_noaug



✅ CHECKPOINT FOUND: MSMP | Random | No Aug


### MSMP | Random | Aug

In [59]:
r_c2_random_aug = load_or_run(
    "c2_random_aug", "random", True,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Random | Aug")
c2_random_aug_cls,c2_random_aug_reg,c2_random_aug_cls_s,c2_random_aug_reg_s,c2_random_aug_pl,c2_random_aug_sim,c2_random_aug_hist = r_c2_random_aug



✅ CHECKPOINT FOUND: MSMP | Random | Aug


### MSMP | Scaffold | No Aug

In [60]:
r_c3_scaffold_noaug = load_or_run(
    "c3_scaffold_noaug", "scaffold", False,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Scaffold | No Aug")
c3_scaffold_noaug_cls,c3_scaffold_noaug_reg,c3_scaffold_noaug_cls_s,c3_scaffold_noaug_reg_s,c3_scaffold_noaug_pl,c3_scaffold_noaug_sim,c3_scaffold_noaug_hist = r_c3_scaffold_noaug



✅ CHECKPOINT FOUND: MSMP | Scaffold | No Aug


### MSMP | Scaffold | Aug

In [61]:
r_c4_scaffold_aug = load_or_run(
    "c4_scaffold_aug", "scaffold", True,
    MODEL_CFG["MSMP"][0], MODEL_CFG["MSMP"][1],
    "MSMP | Scaffold | Aug")
c4_scaffold_aug_cls,c4_scaffold_aug_reg,c4_scaffold_aug_cls_s,c4_scaffold_aug_reg_s,c4_scaffold_aug_pl,c4_scaffold_aug_sim,c4_scaffold_aug_hist = r_c4_scaffold_aug



✅ CHECKPOINT FOUND: MSMP | Scaffold | Aug


### GCN | Random | No Aug

In [62]:
r_gcn_random_noaug = load_or_run(
    "gcn_random_noaug", "random", False,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Random | No Aug")
gcn_random_noaug_cls,gcn_random_noaug_reg,gcn_random_noaug_cls_s,gcn_random_noaug_reg_s,gcn_random_noaug_pl,gcn_random_noaug_sim,gcn_random_noaug_hist = r_gcn_random_noaug



✅ CHECKPOINT FOUND: GCN | Random | No Aug


### GCN | Random | Aug

In [63]:
r_gcn_random_aug = load_or_run(
    "gcn_random_aug", "random", True,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Random | Aug")
gcn_random_aug_cls,gcn_random_aug_reg,gcn_random_aug_cls_s,gcn_random_aug_reg_s,gcn_random_aug_pl,gcn_random_aug_sim,gcn_random_aug_hist = r_gcn_random_aug



✅ CHECKPOINT FOUND: GCN | Random | Aug


### GCN | Scaffold | No Aug

In [64]:
r_gcn_scaffold_noaug = load_or_run(
    "gcn_scaffold_noaug", "scaffold", False,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Scaffold | No Aug")
gcn_scaffold_noaug_cls,gcn_scaffold_noaug_reg,gcn_scaffold_noaug_cls_s,gcn_scaffold_noaug_reg_s,gcn_scaffold_noaug_pl,gcn_scaffold_noaug_sim,gcn_scaffold_noaug_hist = r_gcn_scaffold_noaug



✅ CHECKPOINT FOUND: GCN | Scaffold | No Aug


### GCN | Scaffold | Aug

In [65]:
r_gcn_scaffold_aug = load_or_run(
    "gcn_scaffold_aug", "scaffold", True,
    MODEL_CFG["GCN"][0], MODEL_CFG["GCN"][1],
    "GCN | Scaffold | Aug")
gcn_scaffold_aug_cls,gcn_scaffold_aug_reg,gcn_scaffold_aug_cls_s,gcn_scaffold_aug_reg_s,gcn_scaffold_aug_pl,gcn_scaffold_aug_sim,gcn_scaffold_aug_hist = r_gcn_scaffold_aug



✅ CHECKPOINT FOUND: GCN | Scaffold | Aug


### GAT | Random | No Aug

In [66]:
r_gat_random_noaug = load_or_run(
    "gat_random_noaug", "random", False,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Random | No Aug")
gat_random_noaug_cls,gat_random_noaug_reg,gat_random_noaug_cls_s,gat_random_noaug_reg_s,gat_random_noaug_pl,gat_random_noaug_sim,gat_random_noaug_hist = r_gat_random_noaug



✅ CHECKPOINT FOUND: GAT | Random | No Aug


### GAT | Random | Aug

In [67]:
r_gat_random_aug = load_or_run(
    "gat_random_aug", "random", True,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Random | Aug")
gat_random_aug_cls,gat_random_aug_reg,gat_random_aug_cls_s,gat_random_aug_reg_s,gat_random_aug_pl,gat_random_aug_sim,gat_random_aug_hist = r_gat_random_aug



✅ CHECKPOINT FOUND: GAT | Random | Aug


### GAT | Scaffold | No Aug

In [68]:
r_gat_scaffold_noaug = load_or_run(
    "gat_scaffold_noaug", "scaffold", False,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Scaffold | No Aug")
gat_scaffold_noaug_cls,gat_scaffold_noaug_reg,gat_scaffold_noaug_cls_s,gat_scaffold_noaug_reg_s,gat_scaffold_noaug_pl,gat_scaffold_noaug_sim,gat_scaffold_noaug_hist = r_gat_scaffold_noaug



✅ CHECKPOINT FOUND: GAT | Scaffold | No Aug


### GAT | Scaffold | Aug

In [69]:
r_gat_scaffold_aug = load_or_run(
    "gat_scaffold_aug", "scaffold", True,
    MODEL_CFG["GAT"][0], MODEL_CFG["GAT"][1],
    "GAT | Scaffold | Aug")
gat_scaffold_aug_cls,gat_scaffold_aug_reg,gat_scaffold_aug_cls_s,gat_scaffold_aug_reg_s,gat_scaffold_aug_pl,gat_scaffold_aug_sim,gat_scaffold_aug_hist = r_gat_scaffold_aug



✅ CHECKPOINT FOUND: GAT | Scaffold | Aug


### GT | Random | No Aug

In [70]:
r_gt_random_noaug = load_or_run(
    "gt_random_noaug", "random", False,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Random | No Aug")
gt_random_noaug_cls,gt_random_noaug_reg,gt_random_noaug_cls_s,gt_random_noaug_reg_s,gt_random_noaug_pl,gt_random_noaug_sim,gt_random_noaug_hist = r_gt_random_noaug



✅ CHECKPOINT FOUND: GT | Random | No Aug


### GT | Random | Aug

In [71]:
r_gt_random_aug = load_or_run(
    "gt_random_aug", "random", True,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Random | Aug")
gt_random_aug_cls,gt_random_aug_reg,gt_random_aug_cls_s,gt_random_aug_reg_s,gt_random_aug_pl,gt_random_aug_sim,gt_random_aug_hist = r_gt_random_aug



✅ CHECKPOINT FOUND: GT | Random | Aug


### GT | Scaffold | No Aug

In [72]:
r_gt_scaffold_noaug = load_or_run(
    "gt_scaffold_noaug", "scaffold", False,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Scaffold | No Aug")
gt_scaffold_noaug_cls,gt_scaffold_noaug_reg,gt_scaffold_noaug_cls_s,gt_scaffold_noaug_reg_s,gt_scaffold_noaug_pl,gt_scaffold_noaug_sim,gt_scaffold_noaug_hist = r_gt_scaffold_noaug



✅ CHECKPOINT FOUND: GT | Scaffold | No Aug


### GT | Scaffold | Aug

In [73]:
r_gt_scaffold_aug = load_or_run(
    "gt_scaffold_aug", "scaffold", True,
    MODEL_CFG["GT"][0], MODEL_CFG["GT"][1],
    "GT | Scaffold | Aug")
gt_scaffold_aug_cls,gt_scaffold_aug_reg,gt_scaffold_aug_cls_s,gt_scaffold_aug_reg_s,gt_scaffold_aug_pl,gt_scaffold_aug_sim,gt_scaffold_aug_hist = r_gt_scaffold_aug



✅ CHECKPOINT FOUND: GT | Scaffold | Aug


## Cell 30 — Collect All Results

In [74]:
RESULTS = {
    "GCN":  [r_gcn_random_noaug,  r_gcn_random_aug,
             r_gcn_scaffold_noaug,  r_gcn_scaffold_aug],
    "GAT":  [r_gat_random_noaug,  r_gat_random_aug,
             r_gat_scaffold_noaug,  r_gat_scaffold_aug],
    "GT":   [r_gt_random_noaug,   r_gt_random_aug,
             r_gt_scaffold_noaug,   r_gt_scaffold_aug],
    "MSMP": [r_c1_random_noaug,   r_c2_random_aug,
             r_c3_scaffold_noaug,   r_c4_scaffold_aug],
}
# condition index: 0=Random NoAug  1=Random Aug  2=Scaffold NoAug  3=Scaffold Aug

CLS_S  = {m: [RESULTS[m][c][2] for c in range(4)] for m in MODEL_NAMES}
REG_S  = {m: [RESULTS[m][c][3] for c in range(4)] for m in MODEL_NAMES}
PL     = {m: [RESULTS[m][c][4] for c in range(4)] for m in MODEL_NAMES}
SIM    = {m: [RESULTS[m][c][5] for c in range(4)] for m in MODEL_NAMES}
HIST   = {m: [RESULTS[m][c][6] for c in range(4)] for m in MODEL_NAMES}
CLS_FL = {m: [RESULTS[m][c][0] for c in range(4)] for m in MODEL_NAMES}

def get_train_metrics(pl_list, metric):
    vals = []
    for fd in pl_list:
        tp, tl = fd[4], fd[5]
        try:
            if metric == "auc":
                vals.append(roc_auc_score(tl, tp))
            else:
                th_ = np.linspace(.1,.9,81)
                a_  = [accuracy_score(tl,(tp>=t).astype(int)) for t in th_]
                p__ = (tp >= th_[np.argmax(a_)]).astype(int)
                if   metric == "accuracy": vals.append(accuracy_score(tl, p__))
                elif metric == "mcc":      vals.append(matthews_corrcoef(tl, p__))
                elif metric == "f1":       vals.append(f1_score(tl, p__))
        except: pass
    return (float(np.mean(vals)) if vals else 0.,
            float(np.std(vals))  if vals else 0.)

print("✅ Results collected")
print("\n─── Test ACC summary (Aug conditions) ───")
for mn in MODEL_NAMES:
    ra = CLS_S[mn][1]["accuracy"]["mean"]
    sa = CLS_S[mn][3]["accuracy"]["mean"]
    print(f"  {mn:<5}  Random Aug={ra:.4f}   Scaffold Aug={sa:.4f}")


✅ Results collected

─── Test ACC summary (Aug conditions) ───
  GCN    Random Aug=0.8795   Scaffold Aug=0.8565
  GAT    Random Aug=0.8921   Scaffold Aug=0.8690
  GT     Random Aug=0.8942   Scaffold Aug=0.8607
  MSMP   Random Aug=0.8902   Scaffold Aug=0.8616


## Cell 31 — Auto-Select Best Model

In [75]:
# Best model = highest composite score on Scaffold Aug (most rigorous condition)
# Composite: AUC×0.4 + ACC×0.3 + MCC×0.3
print("\n─── Best Model Selection (Scaffold Aug) ───")
best_model_name  = None
best_score       = -1

for mn in MODEL_NAMES:
    auc   = CLS_S[mn][3]["auc"]["mean"]
    acc   = CLS_S[mn][3]["accuracy"]["mean"]
    mcc   = CLS_S[mn][3]["mcc"]["mean"]
    score = 0.4*auc + 0.3*acc + 0.3*mcc
    print(f"  {mn:<5}  AUC={auc:.4f}  ACC={acc:.4f}  MCC={mcc:.4f}"
          f"  → score={score:.4f}")
    if score > best_score:
        best_score = score; best_model_name = mn

best_model_cls_class, best_model_hp = MODEL_CFG[best_model_name]
best_model_pl     = PL[best_model_name][3]   # scaffold aug folds
best_model_cls_s  = CLS_S[best_model_name][3]
best_model_reg_s  = REG_S[best_model_name][3]
best_model_color  = MODEL_COLORS[MODEL_NAMES.index(best_model_name)]

print(f"\n🏆 Best model: {best_model_name}  (score={best_score:.4f})")
print(f"   AUC  = {best_model_cls_s['auc']['mean']:.4f}"
      f" ± {best_model_cls_s['auc']['std']:.4f}")
print(f"   ACC  = {best_model_cls_s['accuracy']['mean']:.4f}"
      f" ± {best_model_cls_s['accuracy']['std']:.4f}")
print(f"   MCC  = {best_model_cls_s['mcc']['mean']:.4f}"
      f" ± {best_model_cls_s['mcc']['std']:.4f}")
print(f"   R²   = {best_model_reg_s['r2']['mean']:.4f}"
      f" ± {best_model_reg_s['r2']['std']:.4f}")
print(f"   RMSE = {best_model_reg_s['rmse']['mean']:.4f}"
      f" ± {best_model_reg_s['rmse']['std']:.4f}")



─── Best Model Selection (Scaffold Aug) ───
  GCN    AUC=0.9214  ACC=0.8565  MCC=0.7135  → score=0.8396
  GAT    AUC=0.9283  ACC=0.8690  MCC=0.7376  → score=0.8533
  GT     AUC=0.9264  ACC=0.8607  MCC=0.7237  → score=0.8458
  MSMP   AUC=0.9260  ACC=0.8616  MCC=0.7235  → score=0.8460

🏆 Best model: GAT  (score=0.8533)
   AUC  = 0.9283 ± 0.0112
   ACC  = 0.8690 ± 0.0145
   MCC  = 0.7376 ± 0.0293
   R²   = 0.6210 ± 0.0256
   RMSE = 0.7406 ± 0.0375


## Cell 32 — Paper Tables (Train & Test)

In [76]:
cond_short = ["Random NoAug","Random Aug","Scaffold NoAug","Scaffold Aug"]

for ci, cname in enumerate(cond_short):
    print(f"\n{'='*94}")
    print(f"  TABLE — {cname}")
    hdr = f"  {'Model':<6} {'AUC':>8} {'SN':>8} {'SP':>8} {'ACC':>8} {'MCC':>8} {'RMSE':>8} {'R2':>8} {'MAE':>8}"
    print("  --- TRAINING ---"); print(hdr)
    for mn in MODEL_NAMES:
        pl = PL[mn][ci]
        ta_auc,ta_sn,ta_sp,ta_acc,ta_mcc,ta_r2,ta_rmse,ta_mae = [],[],[],[],[],[],[],[]
        for fd in pl:
            tp,tl,ttr,tpr = fd[4],fd[5],fd[6],fd[7]  # fd[7] already denormalized ✅
            try:
                ta_auc.append(roc_auc_score(tl, tp))
                th_ = np.linspace(.1,.9,81)
                a_  = [accuracy_score(tl,(tp>=t).astype(int)) for t in th_]
                p__ = (tp >= th_[np.argmax(a_)]).astype(int)
                tn_,fp__,fn_,tp_ = confusion_matrix(tl, p__).ravel()
                ta_acc.append(accuracy_score(tl, p__))
                ta_sn.append(recall_score(tl, p__))
                ta_sp.append(tn_/(tn_+fp__))
                ta_mcc.append(matthews_corrcoef(tl, p__))
                ta_r2.append(r2_score(ttr, tpr))
                ta_rmse.append(np.sqrt(mean_squared_error(ttr, tpr)))
                ta_mae.append(mean_absolute_error(ttr, tpr))
            except: pass
        def mv(lst): return f"{np.mean(lst):.3f}" if lst else "—"
        print(f"  {mn:<6} {mv(ta_auc):>8} {mv(ta_sn):>8} {mv(ta_sp):>8}"
              f" {mv(ta_acc):>8} {mv(ta_mcc):>8}"
              f" {mv(ta_rmse):>8} {mv(ta_r2):>8} {mv(ta_mae):>8}")
    print("  --- TEST ---"); print(hdr)
    for mn in MODEL_NAMES:
        cs = CLS_S[mn][ci]; rs = REG_S[mn][ci]
        print(f"  {mn:<6}"
              f" {cs['auc']['mean']:>8.3f}"
              f" {cs['sensitivity']['mean']:>8.3f}"
              f" {cs['specificity']['mean']:>8.3f}"
              f" {cs['accuracy']['mean']:>8.3f}"
              f" {cs['mcc']['mean']:>8.3f}"
              f" {rs['rmse']['mean']:>8.3f}"
              f" {rs['r2']['mean']:>8.3f}"
              f" {rs['mae']['mean']:>8.3f}")



  TABLE — Random NoAug
  --- TRAINING ---
  Model       AUC       SN       SP      ACC      MCC     RMSE       R2      MAE
  GCN       0.978    0.934    0.902    0.918    0.837    0.572    0.775    0.414
  GAT       0.979    0.941    0.908    0.925    0.850    0.558    0.784    0.405
  GT        0.971    0.921    0.899    0.910    0.821    0.608    0.743    0.440
  MSMP      0.970    0.919    0.906    0.912    0.825    0.626    0.730    0.465
  --- TEST ---
  Model       AUC       SN       SP      ACC      MCC     RMSE       R2      MAE
  GCN       0.941    0.893    0.859    0.876    0.752    0.680    0.687    0.490
  GAT       0.946    0.886    0.883    0.884    0.770    0.673    0.692    0.485
  GT        0.943    0.880    0.877    0.878    0.757    0.686    0.680    0.493
  MSMP      0.946    0.912    0.853    0.882    0.767    0.685    0.682    0.498

  TABLE — Random Aug
  --- TRAINING ---
  Model       AUC       SN       SP      ACC      MCC     RMSE       R2      MAE
  GCN     

## Cell 33 — Statistical Analysis

In [77]:
print("\n─── Aug vs NoAug — paired t-test per model per split ───")
for mn in MODEL_NAMES:
    for si, sname in [(0,"Random"), (2,"Scaffold")]:
        for metric in ["accuracy", "auc", "mcc"]:
            va = [r[metric] for r in CLS_FL[mn][si]]
            vb = [r[metric] for r in CLS_FL[mn][si+1]]
            if len(va) < 2: continue
            t, p = stats.ttest_rel(va, vb)
            sig  = "***" if p<.01 else ("*" if p<.05 else "ns")
            print(f"  {mn} {sname}: NoAug→Aug | {metric}: "
                  f"Δ={np.mean(vb)-np.mean(va):+.4f}"
                  f"  t={t:.3f}  p={p:.3e}  {sig}")

print(f"\n─── All models vs {best_model_name} on Scaffold Aug ───")
for mn in MODEL_NAMES:
    if mn == best_model_name: continue
    for metric in ["accuracy", "auc", "mcc"]:
        vm   = [r[metric] for r in CLS_FL[mn][3]]
        vbest= [r[metric] for r in CLS_FL[best_model_name][3]]
        if len(vm) < 2: continue
        t, p = stats.ttest_rel(vm, vbest)
        sig  = "***" if p<.01 else ("*" if p<.05 else "ns")
        print(f"  {mn} vs {best_model_name} | {metric}: "
              f"Δ={np.mean(vm)-np.mean(vbest):+.4f}"
              f"  t={t:.3f}  p={p:.3e}  {sig}")



─── Aug vs NoAug — paired t-test per model per split ───
  GCN Random: NoAug→Aug | accuracy: Δ=+0.0040  t=-1.142  p=3.171e-01  ns
  GCN Random: NoAug→Aug | auc: Δ=+0.0030  t=-1.325  p=2.559e-01  ns
  GCN Random: NoAug→Aug | mcc: Δ=+0.0086  t=-1.311  p=2.601e-01  ns
  GCN Scaffold: NoAug→Aug | accuracy: Δ=+0.0143  t=-1.523  p=2.024e-01  ns
  GCN Scaffold: NoAug→Aug | auc: Δ=+0.0143  t=-1.915  p=1.281e-01  ns
  GCN Scaffold: NoAug→Aug | mcc: Δ=+0.0272  t=-1.464  p=2.171e-01  ns
  GAT Random: NoAug→Aug | accuracy: Δ=+0.0077  t=-2.750  p=5.137e-02  ns
  GAT Random: NoAug→Aug | auc: Δ=+0.0052  t=-3.843  p=1.841e-02  *
  GAT Random: NoAug→Aug | mcc: Δ=+0.0153  t=-2.662  p=5.630e-02  ns
  GAT Scaffold: NoAug→Aug | accuracy: Δ=+0.0127  t=-3.037  p=3.852e-02  *
  GAT Scaffold: NoAug→Aug | auc: Δ=+0.0094  t=-3.091  p=3.655e-02  *
  GAT Scaffold: NoAug→Aug | mcc: Δ=+0.0249  t=-3.014  p=3.941e-02  *
  GT Random: NoAug→Aug | accuracy: Δ=+0.0160  t=-14.234  p=1.415e-04  ***
  GT Random: NoAug→Aug |

## Figures

### Fig 01 — Chemical Space Analysis
*Already generated in Cell 5 above.*

### Fig 02 — Classification Metrics: All Models (Random Aug vs Scaffold Aug)

In [78]:
metrics_cls = ["accuracy","auc","mcc","sensitivity","specificity",
               "f1","precision","balanced_accuracy"]
fig, axes = plt.subplots(2, 4, figsize=(20, 9)); axes = axes.flatten()
for ax, m in zip(axes, metrics_cls):
    x = np.arange(4); w = .32
    r_m = [CLS_S[mn][1][m]["mean"] for mn in MODEL_NAMES]
    r_s = [CLS_S[mn][1][m]["std"]  for mn in MODEL_NAMES]
    s_m = [CLS_S[mn][3][m]["mean"] for mn in MODEL_NAMES]
    s_s = [CLS_S[mn][3][m]["std"]  for mn in MODEL_NAMES]
    b1 = ax.bar(x-w/2, r_m, w, label="Random Aug",   color="#2980B9", alpha=.88, edgecolor="white")
    b2 = ax.bar(x+w/2, s_m, w, label="Scaffold Aug",  color="#27AE60", alpha=.88, edgecolor="white")
    ax.errorbar(x-w/2, r_m, yerr=r_s, fmt="none", color="black", capsize=3, lw=1.2)
    ax.errorbar(x+w/2, s_m, yerr=s_s, fmt="none", color="black", capsize=3, lw=1.2)
    ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.18); ax.set_ylabel("Score", fontsize=9)
    ax.set_title(m.replace("_"," ").title(), fontweight="bold", fontsize=11)
    for b, v in zip(b1, r_m):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=7, fontweight="bold")
    for b, v in zip(b2, s_m):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=7, fontweight="bold")
    if m == "accuracy":
        best_v = max(max(r_m), max(s_m))
        ax.axhline(best_v, color="red", ls="--", lw=1.6,
                   label=f"Best={best_v:.3f}", zorder=3)
    if ax == axes[0]: ax.legend(fontsize=8, loc="lower right")
plt.suptitle("Classification Metrics — GCN / GAT / GT / MSMP"
             " | Random Aug vs Scaffold Aug",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout(); savefig("02_classification_metrics.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\02_classification_metrics.png


### Fig 03 — Regression Metrics: All Models

In [79]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, m, title in zip(axes, ["r2","rmse","mae"], ["R²","RMSE","MAE"]):
    x = np.arange(4); w = .32
    r_m = [REG_S[mn][1][m]["mean"] for mn in MODEL_NAMES]
    r_s = [REG_S[mn][1][m]["std"]  for mn in MODEL_NAMES]
    s_m = [REG_S[mn][3][m]["mean"] for mn in MODEL_NAMES]
    s_s = [REG_S[mn][3][m]["std"]  for mn in MODEL_NAMES]
    b1 = ax.bar(x-w/2, r_m, w, label="Random Aug",  color="#2980B9", alpha=.88, edgecolor="white")
    b2 = ax.bar(x+w/2, s_m, w, label="Scaffold Aug", color="#27AE60", alpha=.88, edgecolor="white")
    ax.errorbar(x-w/2, r_m, yerr=r_s, fmt="none", color="black", capsize=3, lw=1.2)
    ax.errorbar(x+w/2, s_m, yerr=s_s, fmt="none", color="black", capsize=3, lw=1.2)
    for b, v in zip(b1, r_m):
        ax.text(b.get_x()+b.get_width()/2, v+.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    for b, v in zip(b2, s_m):
        ax.text(b.get_x()+b.get_width()/2, v+.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES, fontsize=10, fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=12); ax.set_ylabel("Score")
    if m == "r2":
        ax.axhline(.75, color="red", ls="--", lw=1.6, label="R²=0.75 target")
    ax.legend(fontsize=9)
plt.suptitle("Regression Metrics — GCN / GAT / GT / MSMP"
             " | Random Aug vs Scaffold Aug",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("03_regression_metrics.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\03_regression_metrics.png


### Fig 04 — MSMP: All 4 Conditions (Classification + ACC reference line)

In [80]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9)); axes = axes.flatten()
for ax, m in zip(axes, metrics_cls):
    means = [CLS_S["MSMP"][c][m]["mean"] for c in range(4)]
    stds  = [CLS_S["MSMP"][c][m]["std"]  for c in range(4)]
    bars  = ax.bar(COND_LABELS, means, color=COND_COLORS, alpha=.88, edgecolor="white")
    ax.errorbar(range(4), means, yerr=stds, fmt="none", color="black", capsize=4, lw=1.5)
    ax.set_ylim(0, 1.18); ax.set_ylabel("Score"); ax.tick_params(axis="x", labelsize=8)
    ax.set_title(m.replace("_"," ").title(), fontweight="bold", fontsize=11)
    for b, v in zip(bars, means):
        ax.text(b.get_x()+b.get_width()/2, v+.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    if m == "accuracy":
        best_acc = CLS_S["MSMP"][1]["accuracy"]["mean"]
        ax.axhline(best_acc, color="red", ls="--", lw=1.6,
                   label=f"Best (R-Aug)={best_acc:.3f}", zorder=3)
        ax.legend(fontsize=8)
plt.suptitle("MSMP — Classification Metrics Across All 4 Conditions",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout(); savefig("04_msmp_classification_conditions.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\04_msmp_classification_conditions.png


### Fig 05 — MSMP: All 4 Conditions (Regression)

In [81]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, m, title in zip(axes, ["r2","rmse","mae"], ["R²","RMSE","MAE"]):
    means = [REG_S["MSMP"][c][m]["mean"] for c in range(4)]
    stds  = [REG_S["MSMP"][c][m]["std"]  for c in range(4)]
    bars  = ax.bar(COND_LABELS, means, color=COND_COLORS, alpha=.88, edgecolor="white")
    ax.errorbar(range(4), means, yerr=stds, fmt="none", color="black", capsize=4, lw=1.5)
    for b, v in zip(bars, means):
        ax.text(b.get_x()+b.get_width()/2, v+.005, f"{v:.3f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=12); ax.set_ylabel("Score")
    if m == "r2":
        ax.axhline(.75, color="red", ls="--", lw=1.6, label="R²=0.75 target")
        ax.legend(fontsize=9)
plt.suptitle("MSMP — Regression Metrics Across All 4 Conditions",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("05_msmp_regression_conditions.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\05_msmp_regression_conditions.png


### Fig 06 — Augmentation Effect Δ(Aug − NoAug) per Model

In [82]:
met_show = ["accuracy","auc","mcc","f1","balanced_accuracy"]
fig, axes = plt.subplots(len(MODEL_NAMES), 2, figsize=(14, 4*len(MODEL_NAMES)))
for row, mn in enumerate(MODEL_NAMES):
    for col, (si, sname, col_hex) in enumerate([
            (0, "Random",   "#1ABC9C"),
            (2, "Scaffold", "#E67E22")]):
        ax = axes[row, col]
        deltas = [CLS_S[mn][si+1][k]["mean"] - CLS_S[mn][si][k]["mean"]
                  for k in met_show]
        ax.barh(met_show, deltas, color=col_hex, edgecolor="white", alpha=.88)
        ax.axvline(0, color="black", lw=1)
        ax.set_xlabel("Δ (Aug − NoAug)")
        ax.set_title(f"{mn} — {sname} Split", fontweight="bold")
        for i, d in enumerate(deltas):
            ax.text(d + (5e-4 if d >= 0 else -5e-4), i, f"{d:+.4f}",
                    va="center", ha="left" if d >= 0 else "right", fontsize=9)
plt.suptitle("Augmentation Effect: Δ(Aug − NoAug) per Model and Split",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("06_augmentation_effect.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\06_augmentation_effect.png


### Fig 07 — Random vs Scaffold per Model

In [83]:
fig, axes = plt.subplots(len(MODEL_NAMES), 2, figsize=(14, 4*len(MODEL_NAMES)))
for row, mn in enumerate(MODEL_NAMES):
    for col, (r_ci, s_ci, aug_label) in enumerate([
            (0, 2, "No Aug"),
            (1, 3, "With Aug")]):
        ax = axes[row, col]
        met4 = ["accuracy","auc","mcc","f1"]; x = np.arange(4); w = .35
        rm  = [CLS_S[mn][r_ci][k]["mean"] for k in met4]
        rs_ = [CLS_S[mn][r_ci][k]["std"]  for k in met4]
        sm  = [CLS_S[mn][s_ci][k]["mean"] for k in met4]
        ss  = [CLS_S[mn][s_ci][k]["std"]  for k in met4]
        b1 = ax.bar(x-w/2, rm, w, label="Random",   color="#2980B9", alpha=.88, edgecolor="white")
        b2 = ax.bar(x+w/2, sm, w, label="Scaffold",  color="#27AE60", alpha=.88, edgecolor="white")
        ax.errorbar(x-w/2, rm, yerr=rs_, fmt="none", color="black", capsize=3, lw=1.2)
        ax.errorbar(x+w/2, sm, yerr=ss,  fmt="none", color="black", capsize=3, lw=1.2)
        ax.set_xticks(x); ax.set_xticklabels(met4)
        ax.set_ylim(0, 1.12); ax.set_ylabel("Score")
        ax.set_title(f"{mn} — {aug_label}", fontweight="bold"); ax.legend(fontsize=8)
plt.suptitle("Random vs Scaffold Split — All Models",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("07_random_vs_scaffold.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\07_random_vs_scaffold.png


### Fig 08 — ROC Curves: Random Split

In [84]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, ci, cond_label in zip(axes, [0, 1],
                               ["No Augmentation", "With Augmentation"]):
    for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]; mfpr = np.linspace(0, 1, 200); tprs, aucs = [], []
        for fd in pl:
            probs, labels = fd[0], fd[1]
            try:
                fpr, tpr, _ = roc_curve(labels, probs)
                aucs.append(roc_auc_score(labels, probs))
                tprs.append(np.interp(mfpr, fpr, tpr))
                ax.plot(fpr, tpr, alpha=.12, color=col, lw=.8)
            except: pass
        if tprs:
            mt = np.mean(tprs, 0); mt[-1] = 1.; st = np.std(tprs, 0)
            ax.plot(mfpr, mt, color=col, lw=2.5,
                    label=f"{mn} (AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f})")
            ax.fill_between(mfpr, np.clip(mt-st,0,1),
                            np.clip(mt+st,0,1), alpha=.10, color=col)
    ax.plot([0,1],[0,1], "k--", lw=1, alpha=.5, label="Random")
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"Random Split — {cond_label}", fontweight="bold", fontsize=12)
    ax.legend(loc="lower right", fontsize=9, framealpha=.9)
    ax.grid(True, alpha=.3)
plt.suptitle("ROC Curves — Random Split | GCN / GAT / GT / MSMP",
             fontweight="bold", fontsize=14)
plt.tight_layout(); savefig("08_roc_random_split.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\08_roc_random_split.png


### Fig 09 — ROC Curves: Scaffold Split

In [85]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, ci, cond_label in zip(axes, [2, 3],
                               ["No Augmentation", "With Augmentation"]):
    for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]; mfpr = np.linspace(0, 1, 200); tprs, aucs = [], []
        for fd in pl:
            probs, labels = fd[0], fd[1]
            try:
                fpr, tpr, _ = roc_curve(labels, probs)
                aucs.append(roc_auc_score(labels, probs))
                tprs.append(np.interp(mfpr, fpr, tpr))
                ax.plot(fpr, tpr, alpha=.12, color=col, lw=.8)
            except: pass
        if tprs:
            mt = np.mean(tprs, 0); mt[-1] = 1.; st = np.std(tprs, 0)
            ax.plot(mfpr, mt, color=col, lw=2.5,
                    label=f"{mn} (AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f})")
            ax.fill_between(mfpr, np.clip(mt-st,0,1),
                            np.clip(mt+st,0,1), alpha=.10, color=col)
    ax.plot([0,1],[0,1], "k--", lw=1, alpha=.5, label="Random")
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"Scaffold Split — {cond_label}", fontweight="bold", fontsize=12)
    ax.legend(loc="lower right", fontsize=9, framealpha=.9)
    ax.grid(True, alpha=.3)
plt.suptitle("ROC Curves — Scaffold Split | GCN / GAT / GT / MSMP",
             fontweight="bold", fontsize=14)
plt.tight_layout(); savefig("09_roc_scaffold_split.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\09_roc_scaffold_split.png


### Fig 10 — Precision-Recall: Random Split

In [86]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, ci, cond_label in zip(axes, [0, 1],
                               ["No Augmentation", "With Augmentation"]):
    for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]; crec = np.linspace(0,1,200); precs, aps = [], []
        for fd in pl:
            probs, labels = fd[0], fd[1]
            try:
                prec, rec, _ = precision_recall_curve(labels, probs)
                aps.append(average_precision_score(labels, probs))
                precs.append(np.interp(crec, rec[::-1], prec[::-1]))
                ax.plot(rec, prec, alpha=.12, color=col, lw=.8)
            except: pass
        if precs:
            mp = np.mean(precs, 0); sp = np.std(precs, 0)
            ax.plot(crec, mp, color=col, lw=2.5,
                    label=f"{mn} (AP={np.mean(aps):.3f}±{np.std(aps):.3f})")
            ax.fill_between(crec, np.clip(mp-sp,0,1),
                            np.clip(mp+sp,0,1), alpha=.10, color=col)
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
    ax.set_title(f"Random Split — {cond_label}", fontweight="bold", fontsize=12)
    ax.legend(loc="lower left", fontsize=9, framealpha=.9)
    ax.grid(True, alpha=.3)
plt.suptitle("Precision-Recall Curves — Random Split | GCN / GAT / GT / MSMP",
             fontweight="bold", fontsize=14)
plt.tight_layout(); savefig("10_pr_random_split.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\10_pr_random_split.png


### Fig 11 — Precision-Recall: Scaffold Split

In [87]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, ci, cond_label in zip(axes, [2, 3],
                               ["No Augmentation", "With Augmentation"]):
    for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
        pl = PL[mn][ci]; crec = np.linspace(0,1,200); precs, aps = [], []
        for fd in pl:
            probs, labels = fd[0], fd[1]
            try:
                prec, rec, _ = precision_recall_curve(labels, probs)
                aps.append(average_precision_score(labels, probs))
                precs.append(np.interp(crec, rec[::-1], prec[::-1]))
                ax.plot(rec, prec, alpha=.12, color=col, lw=.8)
            except: pass
        if precs:
            mp = np.mean(precs, 0); sp = np.std(precs, 0)
            ax.plot(crec, mp, color=col, lw=2.5,
                    label=f"{mn} (AP={np.mean(aps):.3f}±{np.std(aps):.3f})")
            ax.fill_between(crec, np.clip(mp-sp,0,1),
                            np.clip(mp+sp,0,1), alpha=.10, color=col)
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
    ax.set_title(f"Scaffold Split — {cond_label}", fontweight="bold", fontsize=12)
    ax.legend(loc="lower left", fontsize=9, framealpha=.9)
    ax.grid(True, alpha=.3)
plt.suptitle("Precision-Recall Curves — Scaffold Split | GCN / GAT / GT / MSMP",
             fontweight="bold", fontsize=14)
plt.tight_layout(); savefig("11_pr_scaffold_split.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\11_pr_scaffold_split.png


### Fig 12 — Predicted pIC50: Random Split, With Augmentation

In [89]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, mn, col in zip(axes, MODEL_NAMES, MODEL_COLORS):
    pl = PL[mn][1]   # Random Aug
    at = np.concatenate([fd[2] for fd in pl])
    ap = np.concatenate([fd[3] for fd in pl])
    r2   = r2_score(at, ap)
    rmse = float(np.sqrt(mean_squared_error(at, ap)))
    ax.scatter(at, ap, alpha=.35, s=12, color=col)
    lims = [min(at.min(), ap.min())-.3, max(at.max(), ap.max())+.3]
    ax.plot(lims, lims, "k--", lw=1.5, alpha=.6)
    m_, b_ = np.polyfit(at, ap, 1)
    xl = np.linspace(lims[0], lims[1], 200)
    ax.plot(xl, m_*xl+b_, color=col, lw=2, alpha=.85)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("True pIC\u2085\u2080", fontsize=11)
    ax.set_ylabel("Predicted pIC\u2085\u2080", fontsize=11)
    ax.set_title(mn, fontweight="bold", fontsize=18, color=col, pad=10)
    ax.text(0.05, 0.93, "R\u00b2 = {:.4f}\nRMSE = {:.4f}".format(r2, rmse),
            transform=ax.transAxes, fontsize=10, color="dimgray",
            verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="lightgray", alpha=0.85))
plt.suptitle("Predicted vs True pIC\u2085\u2080 — Random Split, With Augmentation",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("12_ic50_random_aug.png")

  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\12_ic50_random_aug.png


### Fig 13 — Predicted pIC50: Scaffold Split, With Augmentation

In [90]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, mn, col in zip(axes, MODEL_NAMES, MODEL_COLORS):
    pl = PL[mn][3]   # Scaffold Aug
    at = np.concatenate([fd[2] for fd in pl])
    ap = np.concatenate([fd[3] for fd in pl])
    r2   = r2_score(at, ap)
    rmse = float(np.sqrt(mean_squared_error(at, ap)))
    ax.scatter(at, ap, alpha=.35, s=12, color=col)
    lims = [min(at.min(), ap.min())-.3, max(at.max(), ap.max())+.3]
    ax.plot(lims, lims, "k--", lw=1.5, alpha=.6)
    m_, b_ = np.polyfit(at, ap, 1)
    xl = np.linspace(lims[0], lims[1], 200)
    ax.plot(xl, m_*xl+b_, color=col, lw=2, alpha=.85)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("True pIC\u2085\u2080", fontsize=11)
    ax.set_ylabel("Predicted pIC\u2085\u2080", fontsize=11)
    ax.set_title(mn, fontweight="bold", fontsize=18, color=col, pad=10)
    ax.text(0.05, 0.93, "R\u00b2 = {:.4f}\nRMSE = {:.4f}".format(r2, rmse),
            transform=ax.transAxes, fontsize=10, color="dimgray",
            verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="lightgray", alpha=0.85))
plt.suptitle("Predicted vs True pIC\u2085\u2080 — Scaffold Split, With Augmentation",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("13_ic50_scaffold_aug.png")

  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\13_ic50_scaffold_aug.png


### Fig 14 — Performance Heatmaps (R² and Accuracy)

In [91]:
r2_mat  = np.array([[np.mean([r2_score(fd[2], fd[3]) for fd in PL[mn][ci]])
                     for ci in range(4)] for mn in MODEL_NAMES])
acc_mat = np.array([[CLS_S[mn][ci]["accuracy"]["mean"]
                     for ci in range(4)] for mn in MODEL_NAMES])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, mat, title, cmap_ in zip(
        axes, [r2_mat, acc_mat],
        ["R² — Mean over 5 Folds", "Accuracy — Mean over 5 Folds"],
        ["YlGn", "Blues"]):
    sns.heatmap(mat, annot=True, fmt=".4f", ax=ax, cmap=cmap_,
                xticklabels=COND_NAMES_LONG, yticklabels=MODEL_NAMES,
                linewidths=.5, linecolor="gray", vmin=.4, vmax=1.)
    bi, bj = np.unravel_index(mat.argmax(), mat.shape)
    ax.add_patch(plt.Rectangle((bj, bi), 1, 1,
                                fill=False, edgecolor="red", lw=3, zorder=5))
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel(
        f"★ Best: {MODEL_NAMES[bi]} | {COND_NAMES_LONG[bj]} = {mat[bi,bj]:.4f}",
        fontsize=10, color="red", fontweight="bold")
    ax.tick_params(axis="x", labelsize=8, rotation=20)
plt.suptitle("Performance Heatmap — Best Cell Highlighted (red border)",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("14_performance_heatmap.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\14_performance_heatmap.png


### Fig 15 — Training Loss Curves

In [92]:
fig, axes = plt.subplots(len(MODEL_NAMES), 4,
                          figsize=(22, 5*len(MODEL_NAMES)))
for row, mn in enumerate(MODEL_NAMES):
    for col, ci in enumerate(range(4)):
        ax = axes[row, col]
        col_hex = MODEL_COLORS[row]
        all_losses = []
        for fold_h in HIST[mn][ci]:
            for seed, h in fold_h:
                all_losses.append([loss for _, loss, _ in h])
        if not all_losses: continue
        maxl = max(len(x) for x in all_losses)
        pad  = np.full((len(all_losses), maxl), np.nan)
        for i, x in enumerate(all_losses):
            for j, v in enumerate(x): pad[i, j] = v
        ml = np.nanmean(pad, 0); sl = np.nanstd(pad, 0)
        ax.plot(ml, color=col_hex, lw=2, label=mn)
        ax.fill_between(range(len(ml)), ml-sl, ml+sl,
                        color=col_hex, alpha=.2)
        ax.set_yscale("log")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title(f"{mn} — {COND_NAMES_LONG[ci]}",
                     fontweight="bold", fontsize=9)
        ax.legend(fontsize=8)
plt.suptitle("Training Loss Curves — All Models & Conditions",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("15_training_loss.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\15_training_loss.png


### Fig 16 — Per-Seed Convergence (Fold 1, Aug Conditions)

In [93]:
fig, axes = plt.subplots(len(MODEL_NAMES), 2,
                          figsize=(16, 5*len(MODEL_NAMES)))
for row, mn in enumerate(MODEL_NAMES):
    for col, (ci, cname) in enumerate([(1,"Random Aug"),(3,"Scaffold Aug")]):
        ax = axes[row, col]
        hist_list = HIST[mn][ci]
        if not hist_list: continue
        fold0 = hist_list[0]   # Fold 1 only
        for si, (seed, h) in enumerate(fold0):
            ax.plot([ep for ep,_,_ in h], [va for _,_,va in h],
                    color=BRIGHT_SEEDS[si % len(BRIGHT_SEEDS)],
                    lw=2., alpha=.85, label=f"seed={seed}")
        ax.set_ylim(.5, 1.)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Validation ACC")
        ax.set_title(f"{mn} — {cname} (Fold 1)",
                     fontweight="bold", fontsize=10)
        ax.legend(fontsize=8, loc="lower right")
plt.suptitle("Per-Seed Convergence Dynamics — Fold 1",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("16_seed_convergence.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\16_seed_convergence.png


### Fig 17 — Confusion Matrices (Random Aug, Best Threshold)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, mn, col in zip(axes, MODEL_NAMES, MODEL_COLORS):
    cms = []
    for fd in PL[mn][1]:
        probs, labels = fd[0], fd[1]
        th = np.linspace(.1,.9,161)
        ac = [accuracy_score(labels,(probs>=t).astype(int)) for t in th]
        cms.append(confusion_matrix(labels,
                                    (probs >= th[np.argmax(ac)]).astype(int)))
    mcm = np.mean(cms, 0)
    sns.heatmap(mcm, annot=True, fmt=".0f", ax=ax,
                cmap=sns.light_palette(col, as_cmap=True),
                xticklabels=["Pred 0","Pred 1"],
                yticklabels=["True 0","True 1"],
                linewidths=.5, linecolor="gray")
    ax.set_title(mn, fontweight="bold", fontsize=14, color=col)
plt.suptitle("Confusion Matrices — Mean over 5 Folds (Random Aug)",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("17_confusion_matrices.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\17_confusion_matrices.png


### Fig 18 — Residuals Analysis (Best Model — auto-selected)

In [ ]:
at  = np.concatenate([fd[2] for fd in best_model_pl])
ap  = np.concatenate([fd[3] for fd in best_model_pl])
res = ap - at

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(at, res, alpha=.4, s=12, color=best_model_color)
axes[0].axhline(0, color="red", ls="--", lw=1.5)
axes[0].set_xlabel("True pIC\u2085\u2080")
axes[0].set_ylabel("Residual (Pred \u2212 True)")
axes[0].set_title(
    "Residual Analysis\nmean={:+.3f}   std={:.3f}".format(res.mean(), res.std()),
    fontweight="bold")

axes[1].hist(res, bins=50, color=best_model_color,
             edgecolor="white", alpha=.88)
axes[1].axvline(res.mean(), color="red", ls="--", lw=1.5,
                label="Mean={:+.3f}".format(res.mean()))
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution", fontweight="bold")
axes[1].legend()

plt.suptitle(
    "Residuals Analysis \u2014 {} (Scaffold Split, With Augmentation)".format(best_model_name),
    fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("18_residuals.png")

  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\18_residuals.png


### Fig 19 — t-SNE Embeddings (Best Model)

In [98]:
print(f"Training t-SNE embedding model ({best_model_name}, 10 epochs)…")
rng  = np.random.RandomState(42)
sidx = rng.choice(len(graphs), size=min(800, len(graphs)), replace=False)
sg   = [graphs[i] for i in sidx]
spic = [float(graphs[i].y_reg.item()) for i in sidx]
sact = [int(graphs[i].y_cls.item())   for i in sidx]

torch.manual_seed(42)
mkw = {"hidden":  best_model_hp["hidden"],
       "heads":   best_model_hp.get("heads", 4),
       "dropout": best_model_hp["dropout"]}
em  = best_model_cls_class(NODE_DIM, DESC_DIM, **mkw).to(device)
eo  = AdamW(em.parameters(), lr=best_model_hp["lr"],
            weight_decay=best_model_hp["weight_decay"])
es  = build_scheduler(eo, best_model_hp["lr"], warmup=3)

all_tr = []
for i in range(len(graphs)):
    all_tr.append(graphs[i])
    if aug_cache[i]: all_tr.append(aug_cache[i][0])
el = DataLoader(all_tr, 192,
                sampler=make_weighted_sampler(all_tr), num_workers=0)
for ep in range(10):
    train_epoch(em, el, eo, es, best_model_hp, device,
                ep, verbose=False, use_aug=True)

em.eval(); embs = []
with torch.no_grad():
    for b in DataLoader(sg, 192, shuffle=False, num_workers=0):
        embs.append(em(b.to(device), return_embedding=True).cpu().numpy())
embs = np.vstack(embs)

try:
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=500)
except TypeError:
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
xy = tsne.fit_transform(embs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc = axes[0].scatter(xy[:,0], xy[:,1], c=spic,
                     cmap="RdYlGn_r", s=18, alpha=.7,
                     edgecolors="white", linewidths=.3)
plt.colorbar(sc, ax=axes[0], label="pIC₅₀")
axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2")
axes[0].set_title("Embeddings coloured by pIC₅₀", fontweight="bold")

for cls, col, lbl in [(0,"#E74C3C","Inactive"),(1,"#3498DB","Active")]:
    mask = np.array(sact) == cls
    axes[1].scatter(xy[mask,0], xy[mask,1], c=col, s=18, alpha=.65,
                    edgecolors="white", linewidths=.3, label=lbl)
axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")
axes[1].set_title("Embeddings coloured by activity", fontweight="bold")
axes[1].legend()

plt.suptitle(f"t-SNE of Learned Graph Embeddings ({best_model_name})",
             fontweight="bold", fontsize=13)
plt.tight_layout(); savefig("19_tsne_embeddings.png")


Training t-SNE embedding model (GAT, 10 epochs)…
  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\19_tsne_embeddings.png


### Fig 20 — Radar Chart (Scaffold Aug)

In [99]:
rkeys  = ["auc","accuracy","mcc","sensitivity","specificity",
           "f1","precision","balanced_accuracy"]
rlabels = ["AUC","ACC","MCC","Sens","Spec","F1","Prec","BalACC"]
N_r    = len(rkeys)
angles = np.linspace(0, 2*np.pi, N_r, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
for mn, col in zip(MODEL_NAMES, MODEL_COLORS):
    vals = [CLS_S[mn][3][k]["mean"] for k in rkeys]
    vals += [vals[0]]
    ax.plot(angles, vals, color=col, lw=2.2, marker="o",
            markersize=5, label=mn)
    ax.fill(angles, vals, color=col, alpha=.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(rlabels, fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.set_yticks([.2,.4,.6,.8,1.])
ax.yaxis.grid(True, alpha=.4); ax.xaxis.grid(True, alpha=.4)
ax.set_title("Model Performance Radar — Scaffold Split, With Augmentation",
             fontweight="bold", fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=11)
plt.tight_layout(); savefig("20_radar_chart.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\20_radar_chart.png


### Fig 21 — Train vs Test Bars (Scaffold Aug)

In [100]:
cmp_met = ["auc","accuracy","mcc","f1"]
cmp_lab = ["AUC-ROC","Accuracy","MCC","F1"]
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, mk, ml in zip(axes.flatten(), cmp_met, cmp_lab):
    x   = np.arange(4); w = .35
    te_m = [CLS_S[mn][3][mk]["mean"] for mn in MODEL_NAMES]
    te_s = [CLS_S[mn][3][mk]["std"]  for mn in MODEL_NAMES]
    tr_m = [get_train_metrics(PL[mn][3], mk)[0] for mn in MODEL_NAMES]
    tr_s = [get_train_metrics(PL[mn][3], mk)[1] for mn in MODEL_NAMES]
    b1 = ax.bar(x-w/2, tr_m, w, label="Train",
                color="#2980B9", alpha=.88, edgecolor="white")
    b2 = ax.bar(x+w/2, te_m, w, label="Test",
                color="#E74C3C", alpha=.88, edgecolor="white")
    ax.errorbar(x-w/2, tr_m, yerr=tr_s,
                fmt="none", color="black", capsize=4, lw=1.3)
    ax.errorbar(x+w/2, te_m, yerr=te_s,
                fmt="none", color="black", capsize=4, lw=1.3)
    for b, v in zip(b1, tr_m):
        ax.text(b.get_x()+b.get_width()/2, v+.008, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8,
                fontweight="bold", color="#1a4f7a")
    for b, v in zip(b2, te_m):
        ax.text(b.get_x()+b.get_width()/2, v+.008, f"{v:.3f}",
                ha="center", va="bottom", fontsize=8,
                fontweight="bold", color="#7b1414")
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_NAMES, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.15); ax.set_ylabel(ml)
    ax.set_title(ml, fontweight="bold")
    ax.legend(fontsize=10); ax.grid(axis="y", alpha=.3)
plt.suptitle("Train vs Test Performance — All Models (Scaffold Split, Aug)",
             fontweight="bold", fontsize=14, y=1.01)
plt.tight_layout(); savefig("21_train_vs_test.png")


  [saved] D:\Research Papers\Actual Biotechnology\figures_v14\21_train_vs_test.png


## External Validation

### Cell EV1 — Load External Dataset

In [101]:
EXT_PATHS = [
    EXT_PATH_DEFAULT,
    r"D:\Biotechnology Research\FDAdrugsGNN.csv",
    r"D:\Research Papers\Biotechnology\FDAdrugsGNN.csv",
    "./FDAdrugsGNN.csv",
    "/content/FDAdrugsGNN.csv",
    "/kaggle/working/FDAdrugsGNN.csv"]
EXT_PATH = next((p for p in EXT_PATHS if os.path.exists(p)), None)
if EXT_PATH is None:
    raise FileNotFoundError(
        "FDAdrugsGNN.csv not found — update EXT_PATH_DEFAULT in Cell 2")
df_ext = pd.read_csv(EXT_PATH)
print(f"External dataset : {EXT_PATH}")
print(f"Shape            : {df_ext.shape}")
print(f"Columns          : {df_ext.columns.tolist()}")
df_ext.head(3)


External dataset : D:\Biotechnology Research\FDAdrugsGNN.csv
Shape            : (80, 535)
Columns          : ['SMILES', 'Name', 'Cls', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'U

,SMILES,Name,Cls,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 525,Unnamed: 526,Unnamed: 527,Unnamed: 528,Unnamed: 529,Unnamed: 530,Unnamed: 531,Unnamed: 532,Unnamed: 533,Unnamed: 534
0,CC1=C(C(=C(C=C1)Cl)NC2=CC=CC=C2C(=O)O)Cl,Meclofenamate,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CC(C1=CC(=CC=C1)OC2=CC=CC=C2)C(=O)O,Fenoprofen,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CC1=C(C2=C(N1C(=O)C3=CC=C(C=C3)Cl)C=CC(=C2)OC)...,Indomethacin,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Cell EV2 — Build External Graphs

In [102]:
def detect_col(df, candidates):
    for c in df.columns:
        if c.lower() in [x.lower() for x in candidates]:
            return c
    return None

smi_col = detect_col(df_ext, ["smiles","smile","canonical_smiles",
                                "structure","SMILES"])
pic_col = detect_col(df_ext, ["pic50","pIC50","pIC50_mean",
                                "pic50_mean","log_ic50"])
act_col = detect_col(df_ext, ["active","activity","label",
                                "class","bioactivity"])
print(f"SMILES col : {smi_col}")
print(f"pIC50  col : {pic_col}")
print(f"Active col : {act_col}")

ext_graphs, ext_indices = [], []
for idx, row in df_ext.iterrows():
    smi = str(row[smi_col]) if smi_col else None
    if not smi or smi == "nan": continue
    pic50 = float(row[pic_col]) if pic_col and pd.notna(row[pic_col]) else 6.
    act   = int(row[act_col])   if act_col and pd.notna(row[act_col]) else int(pic50>=6)
    g = smiles_to_graph(smi, act, pic50)
    if g:
        g.y_reg_norm = (g.y_reg - y_reg_mean) / y_reg_std
        ext_graphs.append(g); ext_indices.append(idx)

df_ext_valid = df_ext.loc[ext_indices].reset_index(drop=True)
ext_true_cls = [int(g.y_cls.item()) for g in ext_graphs]
ext_true_reg = [float(g.y_reg.item()) for g in ext_graphs]
ext_smiles   = df_ext_valid[smi_col].tolist() if smi_col else []
print(f"\nValid external graphs: {len(ext_graphs)}")
if act_col:
    print(f"Class distribution: {df_ext_valid[act_col].value_counts().to_dict()}")


SMILES col : SMILES
pIC50  col : None
Active col : None

Valid external graphs: 80


### Cell EV3 — Retrain Best Model on Full Dataset & Predict

In [103]:
print(f"Retraining {best_model_name} on full dataset (30 epochs)…")
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

mkw = {"hidden":  best_model_hp["hidden"],
       "heads":   best_model_hp.get("heads", 4),
       "dropout": best_model_hp["dropout"]}
best_model_ext = best_model_cls_class(NODE_DIM, DESC_DIM, **mkw).to(device)
opt_ext   = AdamW(best_model_ext.parameters(),
                  lr=best_model_hp["lr"],
                  weight_decay=best_model_hp["weight_decay"])
sched_ext = build_scheduler(opt_ext, best_model_hp["lr"])

all_tr_graphs = []
for i in range(len(graphs)):
    all_tr_graphs.append(graphs[i])
    all_tr_graphs.extend(aug_cache[i])
full_ldr = DataLoader(all_tr_graphs, 192,
                      sampler=make_weighted_sampler(all_tr_graphs),
                      num_workers=0)

for ep in range(30):
    train_epoch(best_model_ext, full_ldr, opt_ext, sched_ext,
                best_model_hp, device, ep, verbose=True, use_aug=True)

ext_probs, ext_regs_norm = tta_predict(
    best_model_ext, ext_smiles, ext_true_cls, ext_true_reg, device)
ext_preds_reg = ext_regs_norm * y_reg_std + y_reg_mean
print(f"\nExternal prediction complete — {len(ext_probs)} compounds")


Retraining GAT on full dataset (30 epochs)…
    ep  0  loss=0.1858  lr=2.80e-04
    ep  5  loss=0.1021  lr=9.94e-04
    ep 10  loss=0.0756  lr=7.94e-04
    ep 15  loss=0.0623  lr=4.22e-04
    ep 20  loss=0.0523  lr=9.55e-05
    ep 25  loss=0.0630  lr=9.98e-04

External prediction complete — 80 compounds


### Cell EV4 — External Validation Metrics

In [104]:
th  = np.linspace(.1, .9, 161)
ac  = [accuracy_score(ext_true_cls, (ext_probs>=t).astype(int)) for t in th]
best_thr_ext = float(th[np.argmax(ac)])
ext_preds_cls = (ext_probs >= best_thr_ext).astype(int)
has_labels = len(set(ext_true_cls)) > 1
has_reg    = pic_col is not None and len(set(ext_true_reg)) > 1

if has_labels:
    ext_acc  = accuracy_score(ext_true_cls, ext_preds_cls)
    ext_auc  = roc_auc_score(ext_true_cls, ext_probs)
    ext_mcc  = matthews_corrcoef(ext_true_cls, ext_preds_cls)
    ext_f1   = f1_score(ext_true_cls, ext_preds_cls)
    ext_sens = recall_score(ext_true_cls, ext_preds_cls)
    tn, fp_, fn, tp_ = confusion_matrix(ext_true_cls, ext_preds_cls).ravel()
    ext_spec = tn / (tn + fp_)
    print(f"External Validation — {best_model_name} (Final Model)")
    print(f"  ACC={ext_acc:.4f}  AUC={ext_auc:.4f}  MCC={ext_mcc:.4f}")
    print(f"  Sensitivity={ext_sens:.4f}  Specificity={ext_spec:.4f}"
          f"  F1={ext_f1:.4f}")

if has_reg:
    ext_r2   = r2_score(ext_true_reg, ext_preds_reg)
    ext_rmse = float(np.sqrt(mean_squared_error(ext_true_reg, ext_preds_reg)))
    ext_mae  = float(mean_absolute_error(ext_true_reg, ext_preds_reg))
    print(f"\n  R²={ext_r2:.4f}  RMSE={ext_rmse:.4f}  MAE={ext_mae:.4f}")


### Fig 22 — External Validation Figures

In [106]:
n_plots = (2 if has_labels else 0) + (1 if has_reg else 0)
if n_plots == 0:
    print("No ground-truth labels found — skipping external figures.")
else:
    fig, axes = plt.subplots(1, max(n_plots, 1), figsize=(6*n_plots, 5))
    if n_plots == 1: axes = [axes]
    ai = 0

    if has_labels:
        fpr, tpr, _ = roc_curve(ext_true_cls, ext_probs)
        axes[ai].plot(fpr, tpr, color=best_model_color, lw=2.5,
                      label="AUC={:.4f}".format(ext_auc))
        axes[ai].plot([0,1],[0,1], "k--", lw=1, alpha=.5)
        axes[ai].set_xlabel("False Positive Rate")
        axes[ai].set_ylabel("True Positive Rate")
        axes[ai].set_title("External Validation \u2014 ROC Curve",
                            fontweight="bold")
        axes[ai].legend(loc="lower right"); ai += 1

        cm_ext = confusion_matrix(ext_true_cls, ext_preds_cls)
        sns.heatmap(cm_ext, annot=True, fmt="d", ax=axes[ai],
                    cmap=sns.light_palette(best_model_color, as_cmap=True),
                    xticklabels=["Pred 0","Pred 1"],
                    yticklabels=["True 0","True 1"],
                    linewidths=.5)
        axes[ai].set_title("External Validation \u2014 Confusion Matrix",
                            fontweight="bold"); ai += 1

    if has_reg:
        at_ext = np.array(ext_true_reg)
        axes[ai].scatter(at_ext, ext_preds_reg, alpha=.5, s=20,
                         color=best_model_color)
        lims = [min(at_ext.min(), ext_preds_reg.min())-.3,
                max(at_ext.max(), ext_preds_reg.max())+.3]
        axes[ai].plot(lims, lims, "k--", lw=1.5, alpha=.6)
        axes[ai].set_xlabel("True pIC\u2085\u2080")
        axes[ai].set_ylabel("Predicted pIC\u2085\u2080")
        axes[ai].set_title(
            "External Validation \u2014 pIC\u2085\u2080 Scatter\nR\u00b2={:.4f}   RMSE={:.4f}".format(ext_r2, ext_rmse),
            fontweight="bold")

    plt.suptitle(
        "External Validation \u2014 {} (Retrained on Full Dataset)".format(best_model_name),
        fontweight="bold", fontsize=13)
    plt.tight_layout(); savefig("22_external_validation.png")

No ground-truth labels found — skipping external figures.


In [107]:
print("\n" + "="*62)
print(f"  ALL DONE — v14 Final")
print(f"  Best model  : {best_model_name} (Scaffold Aug)")
print(f"  Figures     : {FIG_DIR}")
print(f"  Checkpoints : {CKPT_DIR}")
print("="*62)



  ALL DONE — v14 Final
  Best model  : GAT (Scaffold Aug)
  Figures     : D:\Research Papers\Actual Biotechnology\figures_v14
  Checkpoints : D:\Research Papers\Actual Biotechnology\checkpoints_v12
